In [8]:

# Retail Demand Forecasting — Deliverable 2 (UPDATED)
# Objectives:
#  (1) Build weekly SKU demand series (aligned to Monday)
#  (2) Split each SKU series into train / validation / test
#  (3) Select best model per SKU using VALIDATION MAPE
#  (4) Report HELD-OUT TEST MAPE on equal-sized test periods
#  (5) Produce boxplot of test errors to show spread
#  (6) Produce final 12-week forecasts for each SKU
#  (7) Translate demand -> revenue using median SKU price


import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

from tqdm.auto import tqdm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Ensure we can seamlessly import all modular functions from the src directory
sys.path.append(os.path.abspath(".."))

# Import our specialized pipeline tools
from src.tools import load_raw_data, clean_raw_data

warnings.simplefilter("ignore", ConvergenceWarning)

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ----------------------------
# 0) CONFIG
# ----------------------------
DATA_PATH = "online_retail_II.xlsx"

# Weekly alignment
WEEK_FREQ_ASFREQ = "W-MON"

# Final operational forecast horizon
H_FINAL = 12

# Deliverable 2 validation / test design
BLOCK_SIZE = 4       # each block = 4 weeks
VAL_BLOCKS = 3       # 3 x 4 = 12 validation weeks
TEST_BLOCKS = 4      # 4 x 4 = 16 test weeks

VAL_SIZE = BLOCK_SIZE * VAL_BLOCKS
TEST_SIZE = BLOCK_SIZE * TEST_BLOCKS

MIN_TRAIN_WEEKS = 24
MIN_LEN_REQUIRED = MIN_TRAIN_WEEKS + VAL_SIZE + TEST_SIZE

TOP_N_SKUS = 250
RANDOM_SEED = 42
USE_LGB = True

np.random.seed(RANDOM_SEED)

print("Validation weeks:", VAL_SIZE)
print("Test weeks:", TEST_SIZE)
print("Minimum required history:", MIN_LEN_REQUIRED)

Validation weeks: 12
Test weeks: 16
Minimum required history: 52


In [16]:
df['StockCode'].value_counts()

StockCode
85123A    3421
22423     2044
85099B    2012
21212     1920
21232     1713
          ... 
35999        1
84509e       1
85132c       1
47590a       1
21120        1
Name: count, Length: 4251, dtype: int64

In [13]:
df['Customer ID'].value_counts()

Customer ID
14911.0    5570
17841.0    5043
14606.0    3866
14156.0    2648
12748.0    2633
           ... 
15913.0       1
14495.0       1
15273.0       1
15693.0       1
17440.0       1
Name: count, Length: 4312, dtype: int64

In [9]:
# 1) LOAD + CLEAN RAW DATA

dataset_path = os.path.join(PROJECT_ROOT, 'Datasets', DATA_PATH)

df = load_raw_data(dataset_path)

df = clean_raw_data(df)

print("Clean rows:", len(df))
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Unique SKUs:", df["StockCode"].nunique())
print("Unique Countries:", df["Country"].nunique() if "Country" in df.columns else "N/A")

Loading data from: /Users/federicogiorgi/Library/CloudStorage/OneDrive-Personal/Documenti/GitHub/forecasting-retail/Datasets/online_retail_II.xlsx...
Data loaded successfully. Shape: (525461, 8)
Clean rows: 511565
Date range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00
Unique SKUs: 4251
Unique Countries: 40


In [ ]:
# ============================================================
# 2) AGGREGATE TO WEEKLY SKU LEVEL
# ============================================================
weekly_sku = (
    df.groupby(["StockCode", "Week"], as_index=False)
      .agg(
          Quantity=("Quantity", "sum"),
          Revenue=("Revenue", "sum"),
      )
)

# Total historical revenue per SKU
sku_revenue = (
    weekly_sku.groupby("StockCode")["Revenue"]
              .sum()
              .sort_values(ascending=False)
)

# Median historical price per SKU
sku_price = (
    df.groupby("StockCode")["Price"]
      .median()
      .rename("P_typ")
      .reset_index()
)

print("Weekly SKU rows:", len(weekly_sku))
print("Top-5 SKUs by revenue:")
display(sku_revenue.head(5))

In [ ]:
# ============================================================
# 3) BUILD EVALUATION UNIVERSE
#    Top revenue SKUs with enough history for:
#    train + validation + test
# ============================================================
weeks_per_sku = (
    weekly_sku.groupby("StockCode")["Week"]
              .nunique()
              .rename("n_weeks")
              .reset_index()
)

sku_frame = (
    sku_revenue.rename("total_revenue").reset_index()
               .merge(weeks_per_sku, on="StockCode", how="left")
)

eligible_skus = (
    sku_frame[sku_frame["n_weeks"] >= MIN_LEN_REQUIRED]
      .sort_values("total_revenue", ascending=False)
      .head(TOP_N_SKUS)["StockCode"]
      .astype(str)
      .tolist()
)

print("Eligible SKU count:", len(eligible_skus))
print("Minimum weeks required:", MIN_LEN_REQUIRED)

if len(eligible_skus) > 0:
    tmp = sku_frame.set_index("StockCode").loc[eligible_skus, "n_weeks"]
    print("Eligible weeks range:", int(tmp.min()), "to", int(tmp.max()))

In [ ]:
# ============================================================
# 4) BUILD WEEKLY SERIES CACHE
# ============================================================
weekly_sku["Week"] = pd.to_datetime(weekly_sku["Week"])
weekly_sku["StockCode"] = weekly_sku["StockCode"].astype(str)

eligible_set = set(eligible_skus)
sub_weekly = weekly_sku[weekly_sku["StockCode"].isin(eligible_set)].copy()

series_cache = {}
for sku, g in sub_weekly.groupby("StockCode", sort=False):
    s = (
        g.sort_values("Week")
         .set_index("Week")["Quantity"]
         .astype(float)
    )
    series_cache[sku] = s.asfreq(WEEK_FREQ_ASFREQ).fillna(0.0)

def get_series_for_sku(stockcode: str) -> pd.Series:
    return series_cache[str(stockcode)]

print("Cached series:", len(series_cache))

In [ ]:
# ============================================================
# 5) MAPE HELPERS
# ============================================================
def pointwise_ape(y_true, y_pred, eps=1.0):
    """
    Pointwise absolute percentage error on a 0-100 scale.
    eps avoids divide-by-zero issues when actual demand is zero.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return 100.0 * np.abs(y_true - y_pred) / denom

def mape_0_100(y_true, y_pred, eps=1.0):
    return float(np.mean(pointwise_ape(y_true, y_pred, eps=eps)))

In [ ]:
# ============================================================
# 6) TRAIN / VALIDATION / TEST SPLIT
# ============================================================
def split_series_train_val_test(series, val_size=VAL_SIZE, test_size=TEST_SIZE):
    s = series.dropna().astype(float)
    n = len(s)

    if n < MIN_TRAIN_WEEKS + val_size + test_size:
        return None

    train = s.iloc[: n - val_size - test_size]
    val   = s.iloc[n - val_size - test_size : n - test_size]
    test  = s.iloc[n - test_size :]

    return train, val, test

# quick sanity check
example_sku = eligible_skus[0] if len(eligible_skus) > 0 else None
if example_sku is not None:
    split = split_series_train_val_test(get_series_for_sku(example_sku))
    if split is not None:
        tr, va, te = split
        print("Example SKU:", example_sku)
        print("Train weeks:", len(tr), "| Val weeks:", len(va), "| Test weeks:", len(te))

In [ ]:
# ============================================================
# 7) MODEL DEFINITIONS
# ============================================================

# ---- 7.1 Naive
def model_naive(train: pd.Series, horizon: int):
    return np.repeat(float(train.iloc[-1]), horizon)

# ---- 7.2 SARIMA
def model_sarima_safe(train: pd.Series, horizon: int):
    """
    Non-seasonal ARIMA(1,1,1) with fallback safeguards.
    """
    try:
        m = SARIMAX(
            train,
            order=(1, 1, 1),
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fit = m.fit(disp=False, maxiter=50)
        fcst = fit.forecast(steps=horizon).values.astype(float)

        if (not np.isfinite(fcst).all()) or (np.abs(fcst).max() > 1e6):
            return model_naive(train, horizon)

        return np.maximum(0.0, fcst)

    except Exception:
        return model_naive(train, horizon)

# ---- 7.3 LightGBM recursive lag model
if USE_LGB:
    import lightgbm as lgb

    def make_lagged_df(y: pd.Series, lags: int = 8):
        df_l = pd.DataFrame({"y": y.values}, index=y.index)
        for k in range(1, lags + 1):
            df_l[f"lag_{k}"] = df_l["y"].shift(k)
        df_l = df_l.dropna()

        X = df_l[[f"lag_{k}" for k in range(1, lags + 1)]]
        y_target = df_l["y"]
        return X, y_target

    def model_lgb_recursive(train: pd.Series, horizon: int, lags: int = 8):
        X_train, y_train = make_lagged_df(train, lags=lags)

        if len(X_train) < 10:
            return model_naive(train, horizon)

        model = lgb.LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            min_data_in_leaf=2,
            min_data_in_bin=1,
            subsample=1.0,
            colsample_bytree=1.0,
            random_state=RANDOM_SEED,
            verbosity=-1
        )
        model.fit(X_train, y_train)

        history = train.values.astype(float).tolist()
        preds = []

        for _ in range(horizon):
            x = pd.DataFrame(
                [[history[-k] for k in range(1, lags + 1)]],
                columns=[f"lag_{k}" for k in range(1, lags + 1)]
            )
            yhat = float(model.predict(x)[0])
            yhat = max(0.0, yhat)
            preds.append(yhat)
            history.append(yhat)

        return np.array(preds, dtype=float)

# ---- registry
model_registry = {
    "Naive": model_naive,
    "SARIMA": model_sarima_safe,
}
if USE_LGB:
    model_registry["LightGBM"] = lambda tr, hz: model_lgb_recursive(tr, hz, lags=8)

print("Models:", list(model_registry.keys()))

In [ ]:
# ============================================================
# 8) ROLLING BLOCK EVALUATION
#    Used for validation and test in equal-sized periods
# ============================================================
def rolling_block_evaluate(history, future, model_func, block_size=BLOCK_SIZE, eps=1.0):
    """
    Forecast future in equal-sized chronological blocks.

    For each block:
      1) fit model on current history
      2) forecast next block_size weeks
      3) evaluate against actual block
      4) roll forward by appending actuals
    """
    history = history.copy().astype(float)
    future = future.copy().astype(float)

    n = len(future)
    if n % block_size != 0:
        raise ValueError("Future length must be divisible by block_size.")

    rows = []
    n_blocks = n // block_size

    for b in range(n_blocks):
        start = b * block_size
        end = (b + 1) * block_size

        actual_block = future.iloc[start:end]
        fcst = np.asarray(model_func(history, block_size), dtype=float).reshape(-1)

        if len(fcst) != block_size or (not np.isfinite(fcst).all()):
            fcst = np.asarray(model_naive(history, block_size), dtype=float).reshape(-1)

        fcst = np.maximum(0.0, fcst)
        ape = pointwise_ape(actual_block.values, fcst, eps=eps)

        block_df = pd.DataFrame({
            "Block": b + 1,
            "Step": np.arange(1, block_size + 1),
            "Actual": actual_block.values.astype(float),
            "Forecast": fcst.astype(float),
            "APE": ape.astype(float)
        })
        rows.append(block_df)

        # roll forward using actual realized values
        history = pd.concat([history, actual_block])

    out = pd.concat(rows, ignore_index=True)
    return out

In [ ]:
# ============================================================
# 9) VALIDATION-BASED MODEL SELECTION + TEST EVALUATION
#    USING BLOCK-LEVEL MAPE
# ============================================================
selection_rows = []
test_detail_rows = []
validation_detail_rows = []
validation_block_rows = []
test_block_rows = []

for sku in tqdm(eligible_skus, desc="Deliverable 2 evaluation"):
    s = get_series_for_sku(sku)
    split = split_series_train_val_test(s)

    if split is None:
        continue

    train, val, test = split

    # -------------------------
    # Validation: choose model using BLOCK-LEVEL MAPE
    # -------------------------
    val_scores = {}

    for mname, mf in model_registry.items():
        val_eval = rolling_block_evaluate(train, val, mf, block_size=BLOCK_SIZE)
        val_eval["StockCode"] = sku
        val_eval["Model"] = mname
        validation_detail_rows.append(val_eval)

        val_block = (
            val_eval.groupby(["StockCode", "Model", "Block"], as_index=False)
                    .agg(
                        Actual_Block=("Actual", "sum"),
                        Forecast_Block=("Forecast", "sum")
                    )
        )

        val_block["Block_APE"] = (
            100.0 * np.abs(val_block["Actual_Block"] - val_block["Forecast_Block"])
            / np.maximum(np.abs(val_block["Actual_Block"]), 1.0)
        )

        validation_block_rows.append(val_block)
        val_scores[mname] = float(val_block["Block_APE"].mean())

    best_model_name = min(val_scores, key=val_scores.get)

    row = {
        "StockCode": sku,
        "Train_Weeks": len(train),
        "Val_Weeks": len(val),
        "Test_Weeks": len(test),
        "Chosen_Model": best_model_name,
        "Best_Val_Block_MAPE": val_scores[best_model_name]
    }

    for mname in model_registry.keys():
        row[f"Val_Block_MAPE_{mname}"] = val_scores.get(mname, np.nan)

    selection_rows.append(row)

    # -------------------------
    # Test: evaluate chosen model
    # -------------------------
    history_tv = pd.concat([train, val])
    test_eval = rolling_block_evaluate(history_tv, test, model_registry[best_model_name], block_size=BLOCK_SIZE)
    test_eval["StockCode"] = sku
    test_eval["Chosen_Model"] = best_model_name
    test_detail_rows.append(test_eval)

    test_block = (
        test_eval.groupby(["StockCode", "Chosen_Model", "Block"], as_index=False)
                 .agg(
                     Actual_Block=("Actual", "sum"),
                     Forecast_Block=("Forecast", "sum")
                 )
    )

    test_block["Block_APE"] = (
        100.0 * np.abs(test_block["Actual_Block"] - test_block["Forecast_Block"])
        / np.maximum(np.abs(test_block["Actual_Block"]), 1.0)
    )

    test_block_rows.append(test_block)

choices_df = pd.DataFrame(selection_rows)
validation_detail = pd.concat(validation_detail_rows, ignore_index=True)
validation_block = pd.concat(validation_block_rows, ignore_index=True)
test_detail = pd.concat(test_detail_rows, ignore_index=True)
test_block_sku = pd.concat(test_block_rows, ignore_index=True)

print("choices_df shape:", choices_df.shape)
print("validation_detail shape:", validation_detail.shape)
print("validation_block shape:", validation_block.shape)
print("test_detail shape:", test_detail.shape)
print("test_block_sku shape:", test_block_sku.shape)

display(choices_df.head())
display(test_block_sku.head())

In [ ]:
# ============================================================
# 10) DELIVERABLE 2 SUMMARY TABLES (BLOCK-LEVEL)
# ============================================================

overall_test_mape = float(test_block_sku["Block_APE"].mean())

test_block_period_summary = (
    test_block_sku.groupby("Block")["Block_APE"]
                  .agg(
                      Test_MAPE="mean",
                      Median_APE="median",
                      P90_APE=lambda x: np.percentile(x, 90),
                      P95_APE=lambda x: np.percentile(x, 95),
                      N="count"
                  )
                  .reset_index()
)

test_model_period_summary = (
    test_block_sku.groupby(["Chosen_Model", "Block"])["Block_APE"]
                  .agg(
                      Test_MAPE="mean",
                      Median_APE="median",
                      N="count"
                  )
                  .reset_index()
)

val_model_summary = (
    validation_block.groupby("Model")["Block_APE"]
                    .agg(
                        Validation_MAPE="mean",
                        Median_APE="median",
                        N="count"
                    )
                    .reset_index()
                    .sort_values("Validation_MAPE")
)

print("Overall held-out test BLOCK-LEVEL MAPE (0-100):", round(overall_test_mape, 2))
display(test_block_period_summary)
display(val_model_summary)
display(test_model_period_summary)

In [ ]:
# ============================================================
# 11) REQUIRED FIGURES FOR DELIVERABLE 2 (BLOCK-LEVEL)
# ============================================================

# 1) Block-level boxplot by equal-sized test period
plt.figure(figsize=(8, 4))
box_data = [
    test_block_sku.loc[test_block_sku["Block"] == b, "Block_APE"].values
    for b in sorted(test_block_sku["Block"].unique())
]
plt.boxplot(
    box_data,
    tick_labels=[f"Period {b}" for b in sorted(test_block_sku["Block"].unique())],
    showfliers=False
)
plt.title("Block-Level Absolute Percentage Errors on Held-Out Test Set")
plt.xlabel("Equal-sized test period")
plt.ylabel("APE (0-100)")
plt.tight_layout()
plt.show()

# 2) Block-level test MAPE by period
plt.figure(figsize=(8, 4))
plt.bar(
    test_block_period_summary["Block"].astype(str),
    test_block_period_summary["Test_MAPE"].values
)
plt.title("Held-Out Test MAPE by 4-Week Test Period")
plt.xlabel("Test period")
plt.ylabel("MAPE (0-100)")
plt.tight_layout()
plt.show()

# 3) Chosen model counts
plt.figure(figsize=(8, 4))
chosen_counts = choices_df["Chosen_Model"].value_counts()
plt.bar(chosen_counts.index.astype(str), chosen_counts.values)
plt.title("Best Model Selected per SKU (Validation Block-MAPE Criterion)")
plt.xlabel("Chosen model")
plt.ylabel("Number of SKUs")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 12) FINAL 12-WEEK AHEAD FORECASTS
#    After evaluation, refit chosen model on full history
# ============================================================
best_model_map = dict(zip(choices_df["StockCode"].astype(str), choices_df["Chosen_Model"]))

final_rows = []

for sku in choices_df["StockCode"].astype(str):
    s = get_series_for_sku(sku)

    best_model_name = best_model_map.get(sku, "Naive")
    mf = model_registry[best_model_name]

    fcst = np.asarray(mf(s, H_FINAL), dtype=float).reshape(-1)

    if len(fcst) != H_FINAL or (not np.isfinite(fcst).all()):
        best_model_name = "Naive"
        fcst = np.asarray(model_registry["Naive"](s, H_FINAL), dtype=float).reshape(-1)

    fcst = np.maximum(0.0, fcst)

    for h, yhat in enumerate(fcst, start=1):
        final_rows.append((sku, h, float(yhat), best_model_name))

forecast_df = pd.DataFrame(
    final_rows,
    columns=["StockCode", "Horizon", "Forecast", "Chosen_Model"]
)

print("Forecast rows:", len(forecast_df))
print("Unique SKUs forecasted:", forecast_df["StockCode"].nunique())
display(forecast_df.head())

In [ ]:
# ============================================================
# 13) REVENUE TRANSLATION
# ============================================================
fc = forecast_df.copy()
fc["StockCode"] = fc["StockCode"].astype(str)

sku_price2 = sku_price.copy()
sku_price2["StockCode"] = sku_price2["StockCode"].astype(str)

fc = fc.merge(sku_price2, on="StockCode", how="left")
fc["Revenue_Forecast"] = fc["Forecast"] * fc["P_typ"]

# Aggregate projected demand / revenue across 12-week horizon
sku_proj = (
    fc.groupby("StockCode", as_index=False)
      .agg(
          Projected_Demand=("Forecast", "sum"),
          Projected_Revenue=("Revenue_Forecast", "sum"),
          Chosen_Model=("Chosen_Model", lambda x: x.iloc[0])
      )
)

TOP_N = 10

top_by_revenue = sku_proj.sort_values("Projected_Revenue", ascending=False).head(TOP_N)
top_by_demand  = sku_proj.sort_values("Projected_Demand", ascending=False).head(TOP_N)

display(top_by_revenue)
display(top_by_demand)

In [ ]:
# ============================================================
# 14) OPTIONAL: VOLATILITY SEGMENTATION (BLOCK-LEVEL)
#    Useful if you still want an appendix-style diagnostic
# ============================================================
def compute_cv(series: pd.Series):
    mu = float(series.mean())
    sd = float(series.std())
    if mu <= 1e-12:
        return np.nan
    return sd / mu

vol_rows = []
for sku in choices_df["StockCode"].astype(str):
    s = get_series_for_sku(sku)
    cv = compute_cv(s)
    vol_rows.append((sku, cv))

vol_df = pd.DataFrame(vol_rows, columns=["StockCode", "CV"]).dropna()
vol_df["Vol_Group"] = pd.qcut(vol_df["CV"], q=3, labels=["LowVol", "MidVol", "HighVol"])

# Merge onto BLOCK-level test results, not weekly test_detail
test_block_vol = test_block_sku.merge(
    vol_df[["StockCode", "Vol_Group"]],
    on="StockCode",
    how="left"
)

vol_period_summary = (
    test_block_vol.groupby(["Vol_Group", "Block"])["Block_APE"]
                  .agg(
                      Test_MAPE="mean",
                      Median_APE="median",
                      N="count"
                  )
                  .reset_index()
)

display(vol_period_summary.head(12))

In [ ]:
# ============================================================
# 17) BUILD USEFUL OUTPUTS + SAVE SVG/PNG CHARTS
#    Uses the BLOCK-LEVEL Deliverable 2 pipeline
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = "outputs_deliverable2"
os.makedirs(OUT_DIR, exist_ok=True)

import matplotlib.pyplot as plt
plt.rcParams["svg.fonttype"] = "none"

# ----------------------------
# Helper: save fig as BOTH SVG + PNG
# ----------------------------
def save_fig_dual(fig, stem, dpi=220):
    png_path = os.path.join(OUT_DIR, f"{stem}.png")
    svg_path = os.path.join(OUT_DIR, f"{stem}.svg")
    fig.tight_layout()
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight", format="svg")
    plt.close(fig)
    return png_path, svg_path

# ============================================================
# A) CORE VALIDATION / TEST OUTPUTS
# ============================================================

# Overall block-level test MAPE
overall_test_block_mape = float(test_block_sku["Block_APE"].mean())

# Chosen model counts
chosen_counts = (
    choices_df["Chosen_Model"]
    .value_counts()
    .rename_axis("Chosen_Model")
    .reset_index(name="Num_SKUs")
)

# Report-friendly test period summary
test_block_period_summary_report = test_block_period_summary.copy()
test_block_period_summary_report["Test_MAPE"] = test_block_period_summary_report["Test_MAPE"].round(2)
test_block_period_summary_report["Median_APE"] = test_block_period_summary_report["Median_APE"].round(2)
test_block_period_summary_report["P90_APE"] = test_block_period_summary_report["P90_APE"].round(2)
test_block_period_summary_report["P95_APE"] = test_block_period_summary_report["P95_APE"].round(2)

# ============================================================
# B) FORECAST / BUSINESS OUTPUTS
# ============================================================

# ---- Merge forecasts with prices
fc_rev = forecast_df.copy()
fc_rev["StockCode"] = fc_rev["StockCode"].astype(str)

sku_price2 = sku_price.copy()
sku_price2["StockCode"] = sku_price2["StockCode"].astype(str)

fc_rev = fc_rev.merge(sku_price2, on="StockCode", how="left")
fc_rev["Revenue_Forecast"] = fc_rev["Forecast"] * fc_rev["P_typ"]

# ---- Revenue summary by horizon
rev_total = (
    fc_rev.groupby("Horizon", as_index=False)["Revenue_Forecast"]
          .sum()
          .rename(columns={"Revenue_Forecast": "Total_Revenue"})
)

fc_rev["Horizon_Total_Revenue"] = fc_rev.groupby("Horizon")["Revenue_Forecast"].transform("sum")
fc_rev["Revenue_Share"] = fc_rev["Revenue_Forecast"] / fc_rev["Horizon_Total_Revenue"]

top5_share = (
    fc_rev.sort_values(["Horizon", "Revenue_Share"], ascending=[True, False])
          .groupby("Horizon")
          .head(5)
          .groupby("Horizon")["Revenue_Share"]
          .sum()
          .reset_index()
          .rename(columns={"Revenue_Share": "Top5_Share"})
)

hhi = (
    fc_rev.groupby("Horizon")["Revenue_Share"]
          .apply(lambda x: float((x ** 2).sum()))
          .reset_index()
          .rename(columns={"Revenue_Share": "HHI"})
)

rev_summary = (
    rev_total.merge(top5_share, on="Horizon", how="left")
             .merge(hhi, on="Horizon", how="left")
             .sort_values("Horizon")
)

rev_summary_report = rev_summary.copy()
rev_summary_report["Total_Revenue"] = rev_summary_report["Total_Revenue"].map(lambda x: f"${x:,.0f}")
rev_summary_report["Top5_Share"] = rev_summary_report["Top5_Share"].map(lambda x: f"{100*x:.1f}%")
rev_summary_report["HHI"] = rev_summary_report["HHI"].map(lambda x: f"{x:.4f}")

# ---- Top-5 SKUs by revenue within each horizon
top5_by_horizon = (
    fc_rev.sort_values(["Horizon", "Revenue_Forecast"], ascending=[True, False])
          .groupby("Horizon")
          .head(5)
          [["Horizon", "StockCode", "Revenue_Forecast", "Revenue_Share", "Chosen_Model"]]
          .copy()
)
top5_by_horizon["Revenue_Share_pct"] = 100 * top5_by_horizon["Revenue_Share"]

# ---- Aggregate forecast totals by horizon
agg_fc = (
    fc_rev.groupby("Horizon", as_index=False)
          .agg(
              Fc_Total_Demand=("Forecast", "sum"),
              Fc_Total_Revenue=("Revenue_Forecast", "sum")
          )
          .sort_values("Horizon")
)

# ---- Historical weekly aggregate across forecasted SKUs
weekly_hist = weekly_sku[weekly_sku["StockCode"].astype(str).isin(choices_df["StockCode"].astype(str))].copy()
weekly_hist["StockCode"] = weekly_hist["StockCode"].astype(str)
weekly_hist["Week"] = pd.to_datetime(weekly_hist["Week"])

agg_hist = (
    weekly_hist.groupby("Week", as_index=False)
               .agg(
                   Hist_Total_Demand=("Quantity", "sum"),
                   Hist_Total_Revenue=("Revenue", "sum")
               )
               .sort_values("Week")
)

# ---- Continuous no-gap history + forecast extension
last_hist_week = pd.to_datetime(agg_hist["Week"].max())
future_weeks = pd.date_range(
    start=last_hist_week + pd.Timedelta(weeks=1),
    periods=H_FINAL,
    freq="W-MON"
)

fc_cont = agg_fc.copy()
fc_cont["Week"] = future_weeks

agg_demand_cont = pd.concat(
    [
        agg_hist[["Week", "Hist_Total_Demand"]]
            .rename(columns={"Hist_Total_Demand": "Total_Demand"})
            .assign(Series="Historical"),
        fc_cont[["Week", "Fc_Total_Demand"]]
            .rename(columns={"Fc_Total_Demand": "Total_Demand"})
            .assign(Series="Forecast"),
    ],
    ignore_index=True
).sort_values("Week").reset_index(drop=True)

agg_revenue_cont = pd.concat(
    [
        agg_hist[["Week", "Hist_Total_Revenue"]]
            .rename(columns={"Hist_Total_Revenue": "Total_Revenue"})
            .assign(Series="Historical"),
        fc_cont[["Week", "Fc_Total_Revenue"]]
            .rename(columns={"Fc_Total_Revenue": "Total_Revenue"})
            .assign(Series="Forecast"),
    ],
    ignore_index=True
).sort_values("Week").reset_index(drop=True)

# ---- Demand distribution across SKUs
demand_dist = (
    forecast_df.groupby("Horizon")["Forecast"]
               .agg(
                   Median="median",
                   Mean="mean",
                   P10=lambda x: np.percentile(x, 10),
                   P90=lambda x: np.percentile(x, 90)
               )
               .reset_index()
)

# ---- Top projected demand / revenue tables across full 12 weeks
TOP_N = 10

top_by_revenue = (
    sku_proj.sort_values("Projected_Revenue", ascending=False)
            .head(TOP_N)
            .copy()
)

top_by_demand = (
    sku_proj.sort_values("Projected_Demand", ascending=False)
            .head(TOP_N)
            .copy()
)

# ---- Share tables for composition charts
top_by_revenue_chart = top_by_revenue.copy()
top_by_revenue_chart["Revenue_Share_pct"] = (
    100 * top_by_revenue_chart["Projected_Revenue"] / sku_proj["Projected_Revenue"].sum()
)

top_by_demand_chart = top_by_demand.copy()
top_by_demand_chart["Demand_Share_pct"] = (
    100 * top_by_demand_chart["Projected_Demand"] / sku_proj["Projected_Demand"].sum()
)

# ============================================================
# C) OPTIONAL VOLATILITY APPENDIX OUTPUTS (BLOCK-LEVEL)
# ============================================================
def compute_cv(series: pd.Series):
    mu = float(series.mean())
    sd = float(series.std())
    if mu <= 1e-12:
        return np.nan
    return sd / mu

vol_rows = []
for sku in choices_df["StockCode"].astype(str):
    s = get_series_for_sku(sku)
    vol_rows.append((sku, compute_cv(s)))

vol_df = pd.DataFrame(vol_rows, columns=["StockCode", "CV"]).dropna()
vol_df["Vol_Group"] = pd.qcut(vol_df["CV"], q=3, labels=["LowVol", "MidVol", "HighVol"])

test_block_vol = test_block_sku.merge(
    vol_df[["StockCode", "Vol_Group"]],
    on="StockCode",
    how="left"
)

vol_period_summary = (
    test_block_vol.groupby(["Vol_Group", "Block"])["Block_APE"]
                  .agg(
                      Test_MAPE="mean",
                      Median_APE="median",
                      N="count"
                  )
                  .reset_index()
)

# ============================================================
# D) CHARTS
# ============================================================
chart_registry = []

# 1) Block-level boxplot
fig = plt.figure(figsize=(8, 4))
box_data = [
    test_block_sku.loc[test_block_sku["Block"] == b, "Block_APE"].values
    for b in sorted(test_block_sku["Block"].unique())
]
plt.boxplot(
    box_data,
    tick_labels=[f"Period {b}" for b in sorted(test_block_sku["Block"].unique())],
    showfliers=False
)
plt.title("Block-Level Absolute Percentage Errors on Held-Out Test Set")
plt.xlabel("Equal-sized test period")
plt.ylabel("APE (0-100)")
png_path, svg_path = save_fig_dual(fig, "Chart_Block_Boxplot_Test_APE")
chart_registry.append(("Block_Boxplot_Test_APE", png_path, svg_path))

# 2) Held-out test MAPE by 4-week period
fig = plt.figure(figsize=(8, 4))
plt.bar(
    test_block_period_summary["Block"].astype(str),
    test_block_period_summary["Test_MAPE"].values
)
plt.title("Held-Out Test MAPE by 4-Week Test Period")
plt.xlabel("Test period")
plt.ylabel("MAPE (0-100)")
png_path, svg_path = save_fig_dual(fig, "Chart_Block_Test_MAPE_by_Period")
chart_registry.append(("Block_Test_MAPE_by_Period", png_path, svg_path))

# 3) Chosen model counts
fig = plt.figure(figsize=(8, 4))
plt.bar(chosen_counts["Chosen_Model"].astype(str), chosen_counts["Num_SKUs"].values)
plt.title("Best Model Selected per SKU (Validation Block-MAPE Criterion)")
plt.xlabel("Chosen model")
plt.ylabel("Number of SKUs")
png_path, svg_path = save_fig_dual(fig, "Chart_Chosen_Model_Counts")
chart_registry.append(("Chosen_Model_Counts", png_path, svg_path))

# 4) Total demand full history + forecast extension
hist_demand = agg_hist["Hist_Total_Demand"].reset_index(drop=True)
fc_demand = fc_cont["Fc_Total_Demand"].reset_index(drop=True)
combined_weeks = pd.concat([agg_hist["Week"], pd.Series(future_weeks)], ignore_index=True)
combined_demand = pd.concat([hist_demand, fc_demand], ignore_index=True)

fig = plt.figure(figsize=(12, 5))
plt.plot(combined_weeks, combined_demand)
plt.plot(future_weeks, fc_cont["Fc_Total_Demand"], linestyle="--", label="Forecast")
plt.axvline(last_hist_week, color="black", linestyle=":", alpha=0.6)
plt.title("Total Demand — Full History with 12-Week Forecast Extension")
plt.xlabel("Week")
plt.ylabel("Total weekly demand")
plt.legend()
png_path, svg_path = save_fig_dual(fig, "Chart_Total_Demand_FullHistory_Forecast")
chart_registry.append(("Total_Demand_FullHistory_Forecast", png_path, svg_path))

# 5) Portfolio revenue by horizon
fig = plt.figure(figsize=(9, 4))
plt.plot(rev_summary["Horizon"], rev_summary["Total_Revenue"], marker="o")
plt.title("Portfolio Revenue Outlook by Horizon")
plt.xlabel("Forecast horizon (weeks)")
plt.ylabel("Total forecast revenue")
png_path, svg_path = save_fig_dual(fig, "Chart_Portfolio_Revenue_By_Horizon")
chart_registry.append(("Portfolio_Revenue_By_Horizon", png_path, svg_path))

# 6) Concentration metrics by horizon
fig = plt.figure(figsize=(9, 4))
plt.plot(rev_summary["Horizon"], 100 * rev_summary["Top5_Share"], marker="o", label="Top-5 share (%)")
plt.plot(rev_summary["Horizon"], rev_summary["HHI"], marker="o", label="HHI")
plt.title("Revenue Concentration by Horizon")
plt.xlabel("Forecast horizon (weeks)")
plt.ylabel("Metric value")
plt.legend()
png_path, svg_path = save_fig_dual(fig, "Chart_Concentration_By_Horizon")
chart_registry.append(("Concentration_By_Horizon", png_path, svg_path))

# 7) Top-10 projected revenue share
fig = plt.figure(figsize=(10, 5))
plt.bar(top_by_revenue_chart["StockCode"].astype(str), top_by_revenue_chart["Revenue_Share_pct"].values)
plt.title("Top 10 SKUs by Projected Revenue Share (12-Week Horizon)")
plt.xlabel("StockCode")
plt.ylabel("Share of projected revenue (%)")
plt.xticks(rotation=45, ha="right")
png_path, svg_path = save_fig_dual(fig, "Chart_Top10_Projected_Revenue_Share")
chart_registry.append(("Top10_Projected_Revenue_Share", png_path, svg_path))

# 8) Top-10 projected demand share
fig = plt.figure(figsize=(10, 5))
plt.bar(top_by_demand_chart["StockCode"].astype(str), top_by_demand_chart["Demand_Share_pct"].values)
plt.title("Top 10 SKUs by Projected Demand Share (12-Week Horizon)")
plt.xlabel("StockCode")
plt.ylabel("Share of projected demand (%)")
plt.xticks(rotation=45, ha="right")
png_path, svg_path = save_fig_dual(fig, "Chart_Top10_Projected_Demand_Share")
chart_registry.append(("Top10_Projected_Demand_Share", png_path, svg_path))

# 9) Optional volatility appendix chart
fig = plt.figure(figsize=(9, 4))
for vg in ["LowVol", "MidVol", "HighVol"]:
    sub = vol_period_summary[vol_period_summary["Vol_Group"] == vg].sort_values("Block")
    if len(sub) > 0:
        plt.plot(sub["Block"], sub["Test_MAPE"], marker="o", label=vg)
plt.title("Block-Level Test MAPE by Volatility Group")
plt.xlabel("Test period")
plt.ylabel("MAPE (0-100)")
plt.legend()
png_path, svg_path = save_fig_dual(fig, "Chart_Block_Test_MAPE_By_Volatility")
chart_registry.append(("Block_Test_MAPE_By_Volatility", png_path, svg_path))

# Chart manifest
chart_manifest = pd.DataFrame(
    chart_registry,
    columns=["Chart_Name", "PNG_Path", "SVG_Path"]
)

# ============================================================
# E) EXPORT TABLE FILES
# ============================================================
choices_df.to_excel(os.path.join(OUT_DIR, "Model_Choices_Per_SKU_Deliverable2.xlsx"), index=False)
validation_block.to_excel(os.path.join(OUT_DIR, "Validation_Block_Detail_Deliverable2.xlsx"), index=False)
val_model_summary.to_excel(os.path.join(OUT_DIR, "Validation_Model_Summary_Deliverable2.xlsx"), index=False)

test_detail.to_excel(os.path.join(OUT_DIR, "Heldout_Test_Detail_Weekly_Deliverable2.xlsx"), index=False)
test_block_sku.to_excel(os.path.join(OUT_DIR, "Heldout_Test_Block_Detail_Deliverable2.xlsx"), index=False)
test_block_period_summary.to_excel(os.path.join(OUT_DIR, "Heldout_Test_Period_Summary_Deliverable2.xlsx"), index=False)
test_model_period_summary.to_excel(os.path.join(OUT_DIR, "Heldout_Test_Model_Period_Summary_Deliverable2.xlsx"), index=False)
chosen_counts.to_excel(os.path.join(OUT_DIR, "Chosen_Model_Counts_Deliverable2.xlsx"), index=False)

forecast_df.to_excel(os.path.join(OUT_DIR, "Final_Demand_Forecasts_Deliverable2.xlsx"), index=False)
sku_proj.to_excel(os.path.join(OUT_DIR, "SKU_Projected_Revenue_and_Demand_All_Deliverable2.xlsx"), index=False)
top_by_revenue.to_excel(os.path.join(OUT_DIR, f"Top{TOP_N}_SKUs_by_Projected_Revenue_Deliverable2.xlsx"), index=False)
top_by_demand.to_excel(os.path.join(OUT_DIR, f"Top{TOP_N}_SKUs_by_Projected_Demand_Deliverable2.xlsx"), index=False)

rev_summary.to_excel(os.path.join(OUT_DIR, "Portfolio_Revenue_Outlook_By_Horizon.xlsx"), index=False)
rev_summary_report.to_excel(os.path.join(OUT_DIR, "Portfolio_Revenue_Outlook_By_Horizon_ReportFmt.xlsx"), index=False)
top5_by_horizon.to_excel(os.path.join(OUT_DIR, "Top5_SKUs_By_Revenue_Each_Horizon.xlsx"), index=False)

agg_hist.to_excel(os.path.join(OUT_DIR, "Aggregate_History_Deliverable2.xlsx"), index=False)
agg_fc.to_excel(os.path.join(OUT_DIR, "Aggregate_Forecast_Totals_Deliverable2.xlsx"), index=False)
agg_demand_cont.to_excel(os.path.join(OUT_DIR, "Aggregate_Demand_Continuous_Deliverable2.xlsx"), index=False)
agg_revenue_cont.to_excel(os.path.join(OUT_DIR, "Aggregate_Revenue_Continuous_Deliverable2.xlsx"), index=False)
demand_dist.to_excel(os.path.join(OUT_DIR, "Demand_Distribution_Deliverable2.xlsx"), index=False)

vol_df.to_excel(os.path.join(OUT_DIR, "Volatility_CV_Groups_Deliverable2.xlsx"), index=False)
vol_period_summary.to_excel(os.path.join(OUT_DIR, "Volatility_Test_Period_Summary_Deliverable2.xlsx"), index=False)
chart_manifest.to_excel(os.path.join(OUT_DIR, "Chart_Manifest_Deliverable2.xlsx"), index=False)

print("Useful outputs built.")
print("Output folder:", OUT_DIR)
print("Overall held-out block-level MAPE:", round(overall_test_block_mape, 2))
display(test_block_period_summary_report)
display(rev_summary)
display(chosen_counts)

In [ ]:
# ============================================================
# 18) COMPILE ALL OUTPUTS TO ONE EXCEL WORKBOOK
#    Embeds PNG charts; SVG copies remain saved in OUT_DIR
# ============================================================
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.drawing.image import Image as XLImage

compiled_xlsx = os.path.join(OUT_DIR, "Outputs_Compiled_Deliverable2.xlsx")

def add_df_sheet(wb, sheet_name: str, df: pd.DataFrame):
    ws = wb.create_sheet(title=sheet_name[:31])
    out_df = df.copy()

    if isinstance(out_df.index, pd.MultiIndex) or out_df.index.name is not None:
        out_df = out_df.reset_index()

    for r in dataframe_to_rows(out_df, index=False, header=True):
        ws.append(r)

    # simple width sizing
    for col in ws.columns:
        max_len = 0
        col_letter = col[0].column_letter
        for cell in col[:300]:
            if cell.value is None:
                continue
            max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = min(max(10, max_len + 2), 45)

    return ws

wb = Workbook()
wb.remove(wb.active)

# ----------------------------
# 1) README / INDEX
# ----------------------------
ws0 = wb.create_sheet("README")
ws0["A1"] = "Deliverable 2 compiled outputs"
ws0["A3"] = "Core evaluation metric"
ws0["B3"] = "Block-level MAPE on 4 equal-sized held-out test periods"
ws0["A4"] = "Overall held-out block MAPE"
ws0["B4"] = round(overall_test_block_mape, 2)
ws0["A6"] = "Charts"
ws0["A7"] = "PNG charts are embedded in the Charts sheet."
ws0["A8"] = "SVG chart files are saved in the same output folder for report/slide use."
ws0["A10"] = "Output folder"
ws0["B10"] = OUT_DIR

# ----------------------------
# 2) Core evaluation sheets
# ----------------------------
add_df_sheet(wb, "Model_Choices", choices_df)
add_df_sheet(wb, "Chosen_Model_Counts", chosen_counts)
add_df_sheet(wb, "Validation_Block", validation_block)
add_df_sheet(wb, "Validation_Model_Sum", val_model_summary)

# weekly detail kept for appendix transparency
add_df_sheet(wb, "Test_Detail_Weekly", test_detail)

# main deliverable outputs
add_df_sheet(wb, "Test_Block_Detail", test_block_sku)
add_df_sheet(wb, "Test_Period_Summary", test_block_period_summary)
add_df_sheet(wb, "Test_Model_Period", test_model_period_summary)

# ----------------------------
# 3) Forecast / business sheets
# ----------------------------
add_df_sheet(wb, "Final_Forecasts", forecast_df)
add_df_sheet(wb, "SKU_Projected", sku_proj)
add_df_sheet(wb, "Top_Revenue", top_by_revenue)
add_df_sheet(wb, "Top_Demand", top_by_demand)
add_df_sheet(wb, "Revenue_Outlook", rev_summary)
add_df_sheet(wb, "Revenue_Outlook_Fmt", rev_summary_report)
add_df_sheet(wb, "Top5_By_Horizon", top5_by_horizon)

add_df_sheet(wb, "Agg_History", agg_hist)
add_df_sheet(wb, "Agg_Forecast_Totals", agg_fc)
add_df_sheet(wb, "Agg_Demand_Cont", agg_demand_cont)
add_df_sheet(wb, "Agg_Revenue_Cont", agg_revenue_cont)
add_df_sheet(wb, "Demand_Dist", demand_dist)

# ----------------------------
# 4) Optional appendix sheets
# ----------------------------
add_df_sheet(wb, "Volatility_CV", vol_df)
add_df_sheet(wb, "Vol_Test_Period", vol_period_summary)
add_df_sheet(wb, "Chart_Manifest", chart_manifest)

# ----------------------------
# 5) Charts sheet (embed PNGs)
# ----------------------------
wsC = wb.create_sheet("Charts")
wsC["A1"] = "Deliverable 2 charts"
wsC["A2"] = "SVG files are saved in the output folder; PNG versions are embedded below."

row_cursor = 4
for chart_name, png_path, svg_path in chart_registry:
    wsC[f"A{row_cursor}"] = chart_name
    wsC[f"B{row_cursor}"] = os.path.basename(svg_path)
    row_cursor += 1
    try:
        img = XLImage(png_path)
        img.width = img.width * 0.72
        img.height = img.height * 0.72
        wsC.add_image(img, f"A{row_cursor}")
        row_cursor += 22
    except Exception as e:
        wsC[f"A{row_cursor}"] = f"Could not embed {os.path.basename(png_path)}: {e}"
        row_cursor += 2

wb.save(compiled_xlsx)

print("Compiled workbook saved:", compiled_xlsx)
print("SVG charts are also saved in:", OUT_DIR)

In [ ]:
# ============================================================
# 19) BUILD AGENT-READY FORECAST LOOKUP
# ============================================================

agent_lookup = (
    forecast_df.merge(
        sku_price[["StockCode", "P_typ"]].assign(StockCode=lambda x: x["StockCode"].astype(str)),
        on="StockCode",
        how="left"
    )
)

agent_lookup["StockCode"] = agent_lookup["StockCode"].astype(str)
agent_lookup["Revenue_Forecast"] = agent_lookup["Forecast"] * agent_lookup["P_typ"]

# horizon-level rows per SKU
agent_horizon = agent_lookup.copy()

# 12-week aggregate rows per SKU
agent_summary = (
    agent_horizon.groupby("StockCode", as_index=False)
    .agg(
        Chosen_Model=("Chosen_Model", "first"),
        Forecast_12W_Demand=("Forecast", "sum"),
        Forecast_12W_Revenue=("Revenue_Forecast", "sum"),
        Median_Historical_Price=("P_typ", "first")
    )
)

# Merge in evaluation info
agent_summary = agent_summary.merge(
    choices_df[["StockCode", "Best_Val_Block_MAPE", "Chosen_Model"]].rename(
        columns={"Chosen_Model": "Selected_Model_Check"}
    ),
    on="StockCode",
    how="left"
)

agent_summary["StockCode"] = agent_summary["StockCode"].astype(str)

display(agent_summary.head())

In [ ]:
# ============================================================
# 20) SIMPLE PROMPT PARSER
# ============================================================

import re

def extract_sku_from_prompt(prompt: str, valid_skus: set[str]) -> str | None:
    """
    Extract SKU from a free-text prompt.
    Tries exact token matching against known StockCodes.
    """
    prompt = str(prompt).strip()

    # split into alphanumeric tokens
    tokens = re.findall(r"[A-Za-z0-9]+", prompt.upper())
    valid_upper = {sku.upper(): sku for sku in valid_skus}

    for tok in tokens:
        if tok in valid_upper:
            return valid_upper[tok]

    return None

In [ ]:
# ============================================================
# 21) MANAGER-FACING FORECAST RETRIEVAL FUNCTION
# ============================================================

def format_currency(x):
    return f"${x:,.2f}"

def get_latest_forecast_for_sku(prompt: str,
                                agent_summary_df: pd.DataFrame,
                                agent_horizon_df: pd.DataFrame) -> str:
    valid_skus = set(agent_summary_df["StockCode"].astype(str))
    sku = extract_stockcode_from_prompt(prompt, valid_skus)

    if sku is None:
        return (
            "Unable to identify a valid SKU in the prompt. "
            "Please enter a valid StockCode, e.g. '85123A' or '84077'."
        )

    row = agent_summary_df.loc[agent_summary_df["StockCode"] == sku].iloc[0]
    horizon_rows = (
        agent_horizon_df.loc[agent_horizon_df["StockCode"] == sku]
        .sort_values("Horizon")
        [["Horizon", "Forecast", "Revenue_Forecast"]]
    )

    lines = []
    lines.append(f"Forecast retrieval result for SKU {sku}")
    lines.append("-" * 50)
    lines.append(f"Selected model: {row['Chosen_Model']}")
    lines.append(f"Validation block-MAPE: {row['Best_Val_Block_MAPE']:.2f}")
    lines.append(f"12-week projected demand: {row['Forecast_12W_Demand']:.1f}")
    lines.append(f"12-week projected revenue: {format_currency(row['Forecast_12W_Revenue'])}")
    lines.append(f"Median historical price proxy: {format_currency(row['Median_Historical_Price'])}")
    lines.append("")
    lines.append("Weekly forecast breakdown:")

    for _, r in horizon_rows.iterrows():
        lines.append(
            f"Week {int(r['Horizon']):>2}: "
            f"Demand = {r['Forecast']:.1f}, "
            f"Revenue = {format_currency(r['Revenue_Forecast'])}"
        )

    return "\n".join(lines)

In [ ]:
example_prompt_1 = "What is the latest 12-week forecast for product 22197?"
example_prompt_2 = "What is the 12-week demand and revenue outlook for product 84077?"

print(get_latest_forecast_for_sku(example_prompt_1, agent_summary, agent_horizon))
print()
print(get_latest_forecast_for_sku(example_prompt_2, agent_summary, agent_horizon))